### AutoRegressive Properties of Order Book Statistics

The idea is if we provide a set of features X and targets y perhaps we can show that function $y=f(X)$ is invariant to changepoints

In other words the occurrence of a changepoint in X is indicative of a changepoint in y

One way we can do this is individually determine changepoints in both our features and targets and see how close these are -> See if time series model is invariant to changepoints

It would make sense if we didn't solve for autoregressions as this would require the changepoints to occur in very short sequences 

So we want vector regressions between different variables

In [48]:
! pip install ruptures

import ruptures as rpt  # our package
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.api import VAR

from statsmodels.tsa.stattools import adfuller

In [4]:
df=pd.read_csv('/home/jovyan/HFT_Research/PIN_Research/params/hourly_params.csv')
alpha=df['Alpha']
delta=df['Delta']
buys=df['Buy']
sells=df['Sell']
information=df['Mu']
prob=df['prob']

We can solve for an Autoregression model by computing OLS where we regress upon lagged terms and estimate optimal coefficients

We solve for the estimated values along with a two sided confidence interval, can compute p-value signifying if it is noticeable different from zero

In [42]:
adf,p,lag,nobs,cv,icbest=adfuller(prob.dropna())
print("Probability that the Null Hypothesis is true/there is a unit root",p)

Probability that the Null Hypothesis is true/there is a unit root 0.0


In [30]:
res = AutoReg(prob.dropna(),lags=[1,2,3,4,5]).fit()

/opt/conda/lib/python3.8/site-packages/statsmodels/tsa/base/tsa_model.py:213: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  warnings.warn('An unsupported index was provided and will be'


In [31]:
[(res.params.index[i],res.params[i]) for i in range (0,len(res.params))if res.pvalues[i]<0.05 ]

[('intercept', 0.03505922515788118)]

From above we see the PIN does not have an autoregressive structure

In [74]:
model=VAR(df[['Buy','Sell']].dropna()).fit(maxlags=5)

In [75]:
model.summary()

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Fri, 29, Jul, 2022
Time:                     19:42:52
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                   -4.09907
Nobs:                     691.000    HQIC:                  -4.18767
Log likelihood:          -472.825    FPE:                  0.0143566
AIC:                     -4.24355    Det(Omega_mle):       0.0139102
--------------------------------------------------------------------
Results for equation Buy
             coefficient       std. error           t-stat            prob
--------------------------------------------------------------------------
const           0.147030         0.031949            4.602           0.000
L1.Buy          0.382661         0.077062            4.966           0.000
L1.Sell         0.250128         0.074388            3.362           0.001
L2.Buy     

What we could potentially do is perform multivariate time series changepoint detection where we have a vector of

(F(X),Y) -> Where F is the desired on time series model